In [0]:
%sql
USE CATALOG "Workspace";

USE SCHEMA retail_sales;

SELECT
    current_catalog() AS catalog_name,
    current_schema() AS schema_name;

In [0]:
%sql
SHOW TABLES IN retail_sales;

In [0]:
%sql
SELECT
    COUNT(*) AS total_sales_lines,
    ROUND(SUM(net_revenue), 2) AS total_revenue,
    SUM(quantity) AS total_units
FROM silver_store_sales;


In [0]:
%sql
CREATE OR REPLACE TABLE analytics_items
USING DELTA
AS
SELECT
    i.i_item_sk AS item_key,
    i.i_item_id AS item_id,
    i.i_product_name AS product_name,
    i.i_category AS product_category,
    i.i_class AS product_class
FROM samples.tpcds_sf1.item i
INNER JOIN (
    SELECT DISTINCT item_key
    FROM silver_store_sales
    WHERE item_key IS NOT NULL
) s    
ON i.i_item_sk = s.item_key;
    

In [0]:
%sql
CREATE OR REPLACE TABLE analytics_stores
USING DELTA
AS
SELECT
    st.s_store_sk AS store_key,
    st.s_store_id AS store_id,
    st.s_store_name AS store_name,
    st.s_city AS city,
    st.s_state AS state
FROM samples.tpcds_sf1.store st   
INNER JOIN (
    SELECT DISTINCT store_key
    FROM silver_store_sales
    WHERE store_key IS NOT NULL
) s
ON st.s_store_sk = s.store_key;

In [0]:
%sql
CREATE OR REPLACE TABLE analytics_dates
USING DELTA
AS
SELECT
    d.d_date_sk AS date_key,
    CAST(d.d_date AS DATE) AS calendar_date,
    d.d_year AS sales_year,
    d.d_moy AS sales_month,
    d.d_qoy AS sales_quarter
FROM samples.tpcds_sf1.date_dim d  
INNER JOIN (
    SELECT DISTINCT date_key
    FROM silver_store_sales
    WHERE date_key IS NOT NULL
) s   
ON d.d_date_sk = s.date_key;

In [0]:
%sql
CREATE OR REPLACE TABLE analytics_customers
USING DELTA
AS
SELECT
    c.c_customer_sk AS customer_key,
    c.c_customer_id AS customer_id,
    c.c_birth_year AS birth_year
FROM samples.tpcds_sf1.customer c
INNER JOIN (
    SELECT DISTINCT customer_key
    FROM silver_store_sales
    WHERE customer_key IS NOT NULL
) s   
ON c.c_customer_sk = s.customer_key;

In [0]:
%sql
SELECT
    'Products' AS dimension_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT item_key) AS unique_keys
FROM analytics_items

UNION ALL

SELECT
    'Stores',
    COUNT(*),
    COUNT(DISTINCT store_key)
FROM analytics_stores

UNION ALL

SELECT
    'Dates',
    COUNT(*),
    COUNT(DISTINCT date_key)
FROM analytics_dates

UNION ALL

SELECT
    'Customers',
    COUNT(*),
    COUNT(DISTINCT customer_key)
FROM analytics_customers;


In [0]:
%sql
SELECT COUNT(*) AS unmatched_product_rows
FROM silver_store_sales s   
LEFT ANTI JOIN analytics_items i   
    ON s.item_key = i.item_key
WHERE s.item_key IS NOT NULL;

In [0]:
%sql
CREATE OR REPLACE VIEW analytics_sales_enriched AS

SELECT 
    s.date_key,
    d.calendar_date,
    d.sales_year,
    d.sales_month,
    d.sales_quarter,

    s.item_key,
    COALESCE(
        i.product_name,
        'Unknown Product'
    ) AS product_name,

    COALESCE(
        i.product_category,
        'Unknown Category'
    ) AS product_category,

    s.store_key,
    COALESCE(
        st.store_name,
        'Unknown Store'
    ) AS store_name,

    st.city,
    st.state,

    s.customer_key,
    s.ticket_number,
    s.quantity,
    s.sales_price,
    s.net_revenue

FROM silver_store_sales s   

LEFT JOIN analytics_items i   
    ON s.item_key = i.item_key

LEFT JOIN analytics_stores st   
    ON s.store_key = st.store_key

LEFT JOIN analytics_dates d   
    ON s.date_key = d.date_key

LEFT JOIN analytics_customers c   
    ON s.customer_key = c.customer_key;

In [0]:
%sql
SELECT
    'Silver' AS dataset,
    COUNT(*) AS row_count,
    ROUND(SUM(net_revenue), 2) AS total_revenue
FROM silver_store_sales

UNION ALL

SELECT
    'Enriched' AS dataset,
    COUNT(*) AS row_count,
    ROUND(SUM(net_revenue), 2) AS total_revenue
FROM analytics_sales_enriched;

In [0]:
%sql
SELECT
    COUNT(*) AS total_sales_lines,

    SUM(quantity) AS total_units_sold,

    ROUND(
        SUM(net_revenue), 2
    ) AS total_revenue,

    ROUND(
        AVG(net_revenue), 2
    ) AS average_sales_lines_value,

    COUNT(DISTINCT customer_key) AS unique_customers,

    COUNT(DISTINCT store_key) AS total_scores,

    COUNT(DISTINCT item_key) AS unique_products

FROM analytics_sales_enriched;

In [0]:
%sql
SELECT
    DATE_TRUNC(
        'MONTH', calendar_date
    ) AS sales_month,

    COUNT(*) AS sales_lines,

    SUM(quantity) AS units_sold,

    ROUND(
        SUM(net_revenue), 2
    ) AS monthly_revenue

FROM analytics_sales_enriched

WHERE calendar_date IS NOT NULL

GROUP BY
    DATE_TRUNC('MONTH', calendar_date)

ORDER BY sales_month;

In [0]:
%sql
WITH monthly_sales AS (
    SELECT 
        CAST(
            DATE_TRUNC('MONTH', calendar_date)
            AS DATE
        ) AS sales_month,
        SUM(net_revenue) AS revenue
    FROM analytics_sales_enriched
    WHERE calendar_date IS NOT NULL
    GROUP BY 1
),

previous_month AS (
    SELECT
        sales_month,
        revenue,
        LAG(revenue) OVER (
            ORDER BY sales_month
        ) AS previous_revenue,
        LAG(sales_month) OVER (
            ORDER BY sales_month
        ) AS previous_month_date
    FROM monthly_sales
)

SELECT 
    sales_month,
    ROUND(revenue, 2) AS revenue,
    ROUND(previous_revenue, 2) AS previous_revenue,

    CASE
        WHEN previous_month_date = ADD_MONTHS(
            sales_month, -1
        )
        AND previous_revenue > 0
        THEN ROUND(
            (revenue - previous_revenue)
            / previous_revenue * 100,
            2
        )
        ELSE null
    END AS growth_pct
FROM previous_month
ORDER BY sales_month;

In [0]:
%sql
SELECT
    product_category,

    COUNT(*) AS sales_lines,

    SUM(quantity) AS units_sold,

    ROUND(
        SUM(net_revenue), 2
    ) AS total_revenue,

    ROUND(
        AVG(net_revenue), 2
    ) AS average_line_revenue

FROM analytics_sales_enriched

GROUP BY product_category

ORDER BY total_revenue DESC; 

In [0]:
%sql
WITH category_revenue AS (
    SELECT
        product_category,
        SUM(net_revenue) AS revenue
    FROM analytics_sales_enriched
    GROUP BY product_category
)

SELECT
    product_category,
    ROUND(revenue, 2) AS revenue,

    ROUND(
        100.0 * revenue
        / NULLIF(SUM(revenue) OVER (), 0),
        2
    ) AS revenue_share_pct

FROM category_revenue
ORDER BY revenue DESC;

In [0]:
%sql
SELECT
    item_key,
    product_name,
    product_category,

    SUM(quantity) AS units_sold,

    ROUND(
        SUM(net_revenue), 2
    )AS total_revenue

FROM analytics_sales_enriched

GROUP BY
    item_key,
    product_name,
    product_category

ORDER BY total_revenue DESC

LIMIT 10;

In [0]:
%sql
WITH product_sales AS (
    SELECT 
        item_key,
        product_name,
        product_category,
        SUM(net_revenue) AS revenue

    FROM analytics_sales_enriched

    GROUP BY
        item_key,
        product_name,
        product_category
),

ranked_products AS (
    SELECT
        item_key,
        product_name,
        product_category,
        revenue,

        ROW_NUMBER() OVER (
            PARTITION BY product_category
            ORDER BY revenue DESC, item_key
        ) AS product_rank
    FROM product_sales
)

SELECT
    product_category,
    product_name,
    ROUND(revenue, 2) AS revenue,
    product_rank

FROM ranked_products

WHERE product_rank <= 3

ORDER BY
    product_category,
    product_rank;

In [0]:
%sql
WITH store_revenue AS (
    SELECT
        store_key,
        store_name,
        city,
        state,

        SUM(net_revenue) AS revenue,
        SUM(quantity) AS units_sold,
        COUNT(*) AS sales_lines

    FROM analytics_sales_enriched

    GROUP BY
        store_key,
        store_name,
        city,
        state  
)

SELECT
    store_key,
    store_name,
    city,
    state,

    ROUND(revenue, 2) AS revenue,
    units_sold,
    sales_lines,

    RANK() OVER (
        ORDER BY revenue DESC
    ) AS revenue_rank

FROM store_revenue

ORDER BY revenue_rank;

In [0]:
%sql
SELECT 
    customer_key,

    COUNT(*) AS purchase_line_count,

    SUM(quantity) AS total_units,

    ROUND(
        SUM(net_revenue), 2
    ) AS total_spending,

    ROUND(
        AVG(net_revenue), 2
    ) AS average_line_value

FROM analytics_sales_enriched

WHERE customer_key IS NOT NULL

GROUP BY customer_key

ORDER BY total_spending DESC

LIMIT 20;

In [0]:
%sql
WITH customer_spending AS (
    SELECT
        customer_key,
        SUM(net_revenue) AS total_spending

    FROM analytics_sales_enriched

    WHERE customer_key IS NOT NULL

    GROUP BY customer_key
),


segmented_customers AS (
    SELECT
        customer_key,
        total_spending,

        CASE
            WHEN total_spending >= 1000
                THEN 'High Value'

            WHEN total_spending >= 500
                 THEN 'Medium Value'

            ELSE 'Low Value'
        END AS customer_segment

    FROM customer_spending
)

SELECT 
    customer_segment,
    COUNT(*) AS customer_count,

    ROUND(
        SUM(total_spending), 2
    ) AS segment_revenue,

    ROUND(
        AVG(total_spending), 2
    ) AS average_customer_spending

FROM segmented_customers

GROUP BY customer_segment

ORDER BY segment_revenue DESC;


In [0]:
%sql
SELECT 
    COUNT(*) AS total_sales_lines,

    SUM(
        CASE
            WHEN calendar_date IS NULL THEN 1
            ELSE 0
        END   
    ) AS missing_dates,

    SUM(
        CASE
            WHEN product_name = 'Unknown Product'
            THEN 1 ELSE 0
        END  
    ) AS unknown_products,

    SUM(
        CASE
            WHEN store_name = 'Unknown Store'
            THEN 1 ELSE 0
        END 

    ) AS unknown_stores,

    SUM(
        CASE
            WHEN customer_key IS NULL 
            THEN 1 ELSE 0
        END 
    ) AS anonymous_customer_rows

FROM analytics_sales_enriched

In [0]:
%sql
WITH reconciliation AS (
    SELECT 
        (SELECT COUNT(*)
        FROM silver_store_sales) AS silver_rows,

        (SELECT COUNT(*)
         FROM analytics_sales_enriched) AS analytics_rows,
        
        (SELECT SUM(net_revenue)
         FROM silver_store_sales) AS silver_revenue,
        
        (SELECT SUM(net_revenue)
         FROM analytics_sales_enriched) AS analytics_revenue
)

SELECT
    silver_rows,
    analytics_rows,

    ROUND(silver_revenue, 2) AS silver_revenue,
    ROUND(analytics_revenue, 2) AS analytics_revenue,

    CASE
        WHEN silver_rows = analytics_rows
          AND silver_revenue <=> analytics_revenue
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM reconciliation;

In [0]:
%sql
CREATE OR REPLACE VIEW vw_category_sales_report AS

SELECT
    product_category,

    SUM(quantity) AS total_units,

    ROUND(
        SUM(net_revenue), 2
    ) AS total_revenue,

    COUNT(*) AS sales_lines

FROM analytics_sales_enriched

GROUP BY product_category;

In [0]:
%sql
SELECT *
FROM vw_category_sales_report
ORDER BY total_revenue DESC;

In [0]:
%sql
SELECT *
FROM vw_category_sales_report
ORDER BY total_revenue DESC;